### RAG Pipeline - Data Ingestion to Vector DB Pipeline


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


c:\YTRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
## step 1 - Data Ingestion, so we will read all pdfs inside the directory

from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

def process_all_pdfs(pdf_directory):
    """Process all PDF files in the specified directory and return a list of documents."""
    
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory.")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")

        try:
            # Load the PDF
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add metadata
            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["file_path"] = str(pdf_file)

            all_documents.extend(documents)

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"Processed {len(all_documents)} documents from PDF files.")

    return all_documents


all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files in the directory.
Processing file: app_dev interview Q bank.pdf
Processing file: Data Science Interview Qs.pdf
Processing file: Numpy_Pandas_Assignement.pdf
Processing file: Power Bi Workshop – Assignment Instructions.pdf
Processed 94 documents from PDF files.


In [4]:
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files in the directory.
Processing file: app_dev interview Q bank.pdf
Processing file: Data Science Interview Qs.pdf
Processing file: Numpy_Pandas_Assignement.pdf
Processing file: Power Bi Workshop – Assignment Instructions.pdf
Processed 94 documents from PDF files.


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'app_dev interview Q bank.pdf', 'file_path': '..\\data\\pdf_files\\app_dev interview Q bank.pdf', 'total_pages': 14, 'format': 'PDF 1.4', 'title': 'Comprehensive Programming & App Development Interview Question Bank', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Comprehensive Programming & App \nDevelopment Interview Question Bank \nTotal Questions: 200 \nPart 1: Computational Aptitude & Logical Reasoning \n(Questions 1-40) \n1. A train 240m long passes a pole in 24 seconds. How long will it take to pass a platform 650m \nlong? \nAnswer: Speed = Distance/Time = 240/24 = 10 m/s. Total distance for platform = Train + \nPlatform = 240 + 650 = 890m. Time = 890/10 = 89 seconds. \n2. If A can do a work in 10 days and B in 15 days, how long will they take together? \nAnswer: 1/10 + 1/15 = (

In [6]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks using RecursiveCharacterTextSplitter."""
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, 
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # show example of a chunk

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:500]}")  # Print the first 500 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")    

    return split_docs

    

In [7]:
chunks= split_documents(all_pdf_documents)

Split 94 documents into 164 chunks.

Example chunk:
Content: Comprehensive Programming & App 
Development Interview Question Bank 
Total Questions: 200 
Part 1: Computational Aptitude & Logical Reasoning 
(Questions 1-40) 
1. A train 240m long passes a pole in 24 seconds. How long will it take to pass a platform 650m 
long? 
Answer: Speed = Distance/Time = 240/24 = 10 m/s. Total distance for platform = Train + 
Platform = 240 + 650 = 890m. Time = 890/10 = 89 seconds. 
2. If A can do a work in 10 days and B in 15 days, how long will they take together? 
An
Metadata: {'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'app_dev interview Q bank.pdf', 'file_path': '..\\data\\pdf_files\\app_dev interview Q bank.pdf', 'total_pages': 14, 'format': 'PDF 1.4', 'title': 'Comprehensive Programming & App Development Interview Question Bank', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'pag

### Embedding and VectorStoreDB

In [8]:
## embeddings and vector store will be next steps

import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [9]:
## for embedding write a class

class EmbeddingModel:
    """Handels documnet embedding using SentenceTransformer."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding model.
        
        Args:
            model_name (str): Hugging Face model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model() 


    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model.get_sentence_embedding_dimension()} loaded successfully.")
           
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): List of text strings to embed.
        
        Returns:
            np.ndarray: Array of embeddings for the input texts.
        """
        if not self.model:
            raise ValueError("Model not loaded.")
        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## initialize the embedding model

embedding_model = EmbeddingModel()
embedding_model

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3090.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model 384 loaded successfully.


In [10]:
## vector store class

class VectorStore:
    """Manages a vector store using chromaDB for storing document embeddings and metadata."""

    def __init__(self, collection_name: str="pdf_documents", persist_directory: str= "../data/vector_store"):
        """Initialize the vector store.
    
        Args:
        collection_name (str): Name of the chromaDB collection.
        persist_directory (str): Directory to persist the vector store."""

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None  
        self.initialize_store()


    def initialize_store(self):
        """Initialize the chromaDB client and collection."""
        try:
            # create persist directory if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document chunks and their embeddings."})
            
            print(f"Vector store initialized successfully with collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[ Any], embeddings: np.ndarray):
        """Add documents to the vector store with their embeddings.
    
        Args:
        documents (List[Dict[str, Any]]): List of documents with 'content' and 'metadata'.
        embeddings (np.ndarray): Array of embeddings for the documents."""
        
        if len(documents)!= len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store.")

        # prepare data fro chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding )in enumerate(zip(documents, embeddings)):
            # generate unique ID 
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare meatadata
            metadata = dict(doc.metadata) 
            metadata["doc_index"] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_text.append(doc.page_content)

            # embedding
            embeddings_list.append(embedding)
        
        # Add to chromaDB collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store


Vector store initialized successfully with collection: pdf_documents
existing documents in collection: 282


In [11]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'app_dev interview Q bank.pdf', 'file_path': '..\\data\\pdf_files\\app_dev interview Q bank.pdf', 'total_pages': 14, 'format': 'PDF 1.4', 'title': 'Comprehensive Programming & App Development Interview Question Bank', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Comprehensive Programming & App \nDevelopment Interview Question Bank \nTotal Questions: 200 \nPart 1: Computational Aptitude & Logical Reasoning \n(Questions 1-40) \n1. A train 240m long passes a pole in 24 seconds. How long will it take to pass a platform 650m \nlong? \nAnswer: Speed = Distance/Time = 240/24 = 10 m/s. Total distance for platform = Train + \nPlatform = 240 + 650 = 890m. Time = 890/10 = 89 seconds. \n2. If A can do a work in 10 days and B in 15 days, how long will they take together? \nAnswer: 1/10 + 1/15 = (

In [12]:
### convert the text to embeddings and add to vector store

texts = [doc.page_content for doc in all_pdf_documents]


## generate embeddings for the document chunks

embeddings =embedding_model.generate_embeddings(texts)

## store in vector store

vector_store.add_documents(all_pdf_documents, embeddings)


Generating embeddings for 94 texts.


Batches: 100%|██████████| 3/3 [00:05<00:00,  1.77s/it]


Generated embeddings with shape: (94, 384)
Adding 94 documents to the vector store.
Successfully added 94 documents to the vector store.
Total documents in collection after addition: 376


### RAG Retrieval Pipeline from Vector Store

In [13]:
### RAG retrieval Pipeline from vector store

class RAGRetriever:
    """Implements a Retrieval-Augmented Generation (RAG) retriever using a vector store."""

    def __init__(self, vector_store: VectorStore, embedding_model: EmbeddingModel):
        """Initialize the RAG retriever.
        
        Args:
            vector_store (VectorStore): An instance of the VectorStore class.
            embedding_model (EmbeddingModel): An instance of the EmbeddingModel class.
        """
        self.vector_store = vector_store
        self.embedding_model = embedding_model

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents from the vector store based on the query.
        
        Args:
            query (str): The input query string.
            top_k (int): The number of top relevant documents to retrieve.
            score_threshold (float): The minimum similarity score for retrieved documents.

        Returns:
            List[Dict[str, Any]]: A list of retrieved documents with their metadata and similarity scores.
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Generate embedding for the query
        query_embedding = self.embedding_model.generate_embeddings([query])[0]

        #search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            # process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0] 
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                        print(f"Retrieved {len(retrieved_docs)} documnets (after filtering)")
                    else:
                        print("No documenst found")
                    
                    return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise


rag_retriever = RAGRetriever(vector_store, embedding_model)
rag_retriever

In [14]:
rag_retriever.retrieve("What is powerbi dashboard?")

Retrieving documents for query: 'What is powerbi dashboard?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.68it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documnets (after filtering)


[{'id': 'doc_1f156311_93',
  'content': 'Clean and professional presentation\nThis assignment is meant for learning, not perfection.\nNeed Help?\nIf you face issues:\nRe-watch the workshop sessions\nCheck the examples shared during the workshop\nAsk questions during the follow-up discussion on the G-space\nBest of luck, and enjoy building your Power BI dashboard!\n• \n• \n• \n• \n4',
  'metadata': {'creator': 'ChatGPT',
   'title': 'Power Bi Workshop – Assignment Instructions',
   'producer': 'WeasyPrint 65.1',
   'moddate': '',
   'creationDate': '',
   'subject': '',
   'page': 3,
   'trapped': '',
   'doc_index': 93,
   'keywords': '',
   'format': 'PDF 1.7',
   'total_pages': 4,
   'file_path': '..\\data\\pdf_files\\Power Bi Workshop – Assignment Instructions.pdf',
   'source': 'Power Bi Workshop – Assignment Instructions.pdf',
   'author': 'ChatGPT Canvas',
   'creationdate': '',
   'modDate': '',
   'content_length': 331},
  'similarity_score': 0.10924386978149414,
  'distance': 

In [15]:
rag_retriever.retrieve("What is bias, variance trade off ?")

Retrieving documents for query: 'What is bias, variance trade off ?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 63.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documnets (after filtering)


[{'id': 'doc_da0b5cd3_15',
  'content': 'Bias, Variance trade off: \nThe goal of any supervised machine learning algorithm is to have low bias and \nlow variance to achieve good prediction performance. \n1. The k-nearest neighbours algorithm has low bias and high variance, \nbut the trade-off can be changed by increasing the value of k which \nincreases the number of neighbours that contribute to the prediction \nand in turn increases the bias of the model. \n2. The support vector machine algorithm has low bias and high variance, \nbut the trade-off can be changed by increasing the C parameter that \ninfluences the number of violations of the margin allowed in the \ntraining data which increases the bias but decreases the variance. \nThere is no escaping the relationship between bias and variance in machine \nlearning. Increasing the bias will decrease the variance. Increasing the variance \nwill decrease the bias. \n3. What is exploding gradients ? \nGradient: \nGradient is the direct

### Integrate VectorDB Context Pipeline with LLM Outpu

In [16]:
## Simple RAG Pipeline with groq llm

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM(set up API key in .env file)

groq_api_key = os.getenv("GROQ_API_KEY")


llm = ChatGroq(api_key=groq_api_key, model = "llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

###  Simple RAG pipeline function: retrieve context + generate answer

def rag_simple(query, retriever, llm, top_k=int(5)):
    ## retrieve the context
    results= retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant information found in the documents."
    
    ## generate the answer using the GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
    
    Context: {context}

    Question: {query}
    Answer:"""

    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content

    

In [17]:
answer = rag_simple("What is bias variance trade off?", rag_retriever, llm)
print("Answer:", answer)

Retrieving documents for query: 'What is bias variance trade off?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documnets (after filtering)


Answer: The bias-variance tradeoff refers to the fundamental relationship between bias and variance in machine learning, where increasing the bias decreases the variance, and increasing the variance decreases the bias, with the goal of achieving low bias and low variance for good prediction performance.


### Enhance RAG Pipeline

In [18]:
## Enhanced Rag pipeline feature

def rag_advanced(query, retriever, llm, top_k=int(5), min_score=0.2, return_context=False):
   """Rag pipeline with enhanced features:
   -Returns answer, sources, confidence score and opyionally full context."""
   
   results=retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
   if not results:
        return {'answer': 'No relevant context found.', 'sources':[], 'confidence': 0.0, 'context': ''} 

        # prepare context and sources

   context = "\n\n".join([doc['content'] for doc in results])
   sources = [{
    'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
    'page': doc['metadata'].get('page', 'unknown'),
    'score': doc['similarity_score'],
    'preview': doc['content'][:120] + '...'
} for doc in results]

   confidence = max(doc['similarity_score'] for doc in results)

    ## generate the answer using the GROQ LLM
   prompt=f"""Use the following context to answer the question concisely."""
   response = llm.invoke([prompt.format(context=context, query=query)])

   output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
   if return_context:
        output['context'] = context

   return output

In [19]:
result = rag_advanced("What is bias variance trade off?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])    
print("Context Preview:", result['context'][:300])
print("Confidence Score:", result['confidence'])

Retrieving documents for query: 'What is bias variance trade off?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documnets (after filtering)


Answer: You haven't provided the context yet. Please go ahead and provide it, and I'll be happy to help answer your question concisely.
Sources: [{'source': 'Data Science Interview Qs.pdf', 'page': 1, 'score': 0.10429143905639648, 'preview': 'Bias, Variance trade off: \nThe goal of any supervised machine learning algorithm is to have low bias and \nlow variance t...'}]
Context Preview: Bias, Variance trade off: 
The goal of any supervised machine learning algorithm is to have low bias and 
low variance to achieve good prediction performance. 
1. The k-nearest neighbours algorithm has low bias and high variance, 
but the trade-off can be changed by increasing the value of k which 

Confidence Score: 0.10429143905639648


In [20]:

result = rag_advanced("What You Need to Create", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])    
print("Context Preview:", result['context'][:300])
print("Confidence Score:", result['confidence'])

Retrieving documents for query: 'What You Need to Create'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.03it/s]

Generated embeddings with shape: (1, 384)
No documenst found
Answer: No relevant context found.
Sources: []
Context Preview: 
Confidence Score: 0.0


In [24]:
## Advanced RAG Pipeline: [streaming, Citations, History, Summarization, ]

from typing import List, Dict, Any, Tuple
import time

class AdvancedRAGPipeline:
    """Implements an advanced RAG pipeline with features like streaming, citations, history tracking, and summarization."""

    def __init__(self, retriever: RAGRetriever, llm: ChatGroq):
        """Initialize the advanced RAG pipeline.
        
        Args:
            retriever (RAGRetriever): An instance of the RAGRetriever class.
            llm (ChatGroq): An instance of the ChatGroq LLM class.
        """
        self.retriever = retriever
        self.llm = llm
        self.history = []  # To track query and response history

    def query(self, question: str, top_k:int = 5, min_score: float = 0.2, stream : bool = False, summarize: bool = False) -> Dict[str, Any]:
        """Process a query through the RAG pipeline with advanced features.
        
        Args:
            question (str): The input query string.
            top_k (int): The number of top relevant documents to retrieve.
            min_score (float): The minimum similarity score for retrieved documents.
            stream (bool): Whether to stream the response as it's generated.
            summarize (bool): Whether to provide a summary of the retrieved context.

        Returns:
            Dict[str, Any]: A dictionary containing the answer, sources, confidence score, and optionally the context summary.
        """
        # Step 1: Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        
        if not results:
            return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context_summary': ''}

        # Prepare context and sources
        context = "\n\n".join([doc['content'] for doc in results])
        sources = [{
            'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
            'page': doc['metadata'].get('page', 'unknown'),
            'score': doc['similarity_score'],
            'preview': doc['content'][:120] + '...'
        } for doc in results]
        confidence = max(doc['similarity_score'] for doc in results)

        # Optional summarization of context
        context_summary = ""
        if summarize:
            summary_prompt = f"Summarize the following context concisely:\n\n{context}"
            summary_response = self.llm.invoke(summary_prompt)
            context_summary = summary_response.content

        # Generate answer using LLM
        prompt = f"""Use the following context to answer the question concisely.\n\nContext: {context}\n\nQuestion: {question}\nAnswer:"""
        
        if stream:
            # Simulate streaming response by yielding chunks of the answer
            print("Streaming response:")
            for i in range(0, len(prompt), 80):
                print(prompt[i:i+80], end=" ", flush=True)  # Simulate streaming output
                time.sleep(0.05)  # Simulate delay between chunks
            print()
            response = self.llm.invoke(prompt.format(context=context, query=question))
            answer = response.content

    ## Add citations to the answer
        citations =[f"[{i+1}] { src['source']} (page: {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = f"{answer}\n\nSources:\n" + "\n".join(citations) if citations else answer

        # Track history
        self.history.append({
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'confidence': confidence,
            'context_summary': context_summary
        })

        return {
            'answer': answer_with_citations,   
            'sources': sources,
            'confidence': confidence,
            'context_summary': context_summary
        }



## example usage of the advanced RAG pipeline

advanced_rag_pipeline = AdvancedRAGPipeline(rag_retriever, llm)
result = advanced_rag_pipeline.query("What is bias variance trade off?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("Answer with Citations:", result['answer'])   
print("Context Summary:", result['context_summary'])
print("Sources:", result['sources'])
print("Confidence Score:", result['confidence'])
print("Query History:", advanced_rag_pipeline.history)
print("Advanced RAG Pipeline executed successfully.")

Retrieving documents for query: 'What is bias variance trade off?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.04it/s]


Generated embeddings with shape: (1, 384)
Retrieved 1 documnets (after filtering)
Streaming response:
Use the following context to answer the question concisely.

Context: Bias, Vari ance trade off: 
The goal of any supervised machine learning algorithm is to hav e low bias and 
low variance to achieve good prediction performance. 
1. The k-n earest neighbours algorithm has low bias and high variance, 
but the trade-off c an be changed by increasing the value of k which 
increases the number of neighb ours that contribute to the prediction 
and in turn increases the bias of the mo del. 
2. The support vector machine algorithm has low bias and high variance, 
b ut the trade-off can be changed by increasing the C parameter that 
influences t he number of violations of the margin allowed in the 
training data which increa ses the bias but decreases the variance. 
There is no escaping the relationship  between bias and variance in machine 
learning. Increasing the bias will decreas e the v